# Grad-CAM Diagnostic: India False-Positive Shortcut Investigation

**Model:** `efficientnet_b0_forniceal_palpebral_cv` — Fold 4 checkpoint (the fold with the sharpest overfitting and 1.000 India recall, per `CV_5FOLD_ANALYSIS_REPORT.md`).

**Purpose:** test the "visual shortcut vs. calibration artifact" hypothesis for the India cohort's low AUC (~0.60) before committing to any mitigation (HSV+CLAHE, DANN, etc. — see the theoretical consultation earlier in this project). Grad-CAM shows *where* the model is looking when it predicts "anemic" for a true non-anemic India patient. If attention sits on plausible conjunctival tissue, the problem is more likely calibration/threshold. If attention sits on image borders, corners, or diffuse/non-tissue regions, that's direct visual evidence of a shortcut (camera/illumination/compression artifacts, since these `forniceal_palpebral` crops contain no skin or sclera).

**Groups examined:**
1. **India False Positives** (true=Non-anemic, predicted=Anemic) — primary target.
2. **India True Positives** (true=Anemic, predicted=Anemic) — control, to see whether attention differs between correct and incorrect India predictions.
3. **Italy baseline** (correctly classified) — normal model behavior for comparison.

**Runs entirely locally** — no new dependencies beyond what's already in this project's venv (`torch`, `torchvision`, `opencv-python`, `matplotlib`, `pandas`, `numpy`, `Pillow`, `albumentations`, `scikit-learn`). No `pip install` cell needed.


In [1]:
import sys
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image

print("torch:", torch.__version__, " cuda available:", torch.cuda.is_available())


torch: 2.13.0+cu130  cuda available: True


## 1. Project imports

Reuses the shared model builder (`trainer_engine.build_efficientnet_b0`), the shared eval transform
(`dataset.get_eval_transforms` — `Resize(256,256) -> Normalize(ImageNet mean/std) -> ToTensorV2`, matching
exactly what validation used during training), and the CV pipeline's own fold-reconstruction functions
(`cv_dataset.load_cv_pool` / `build_folds`) — calling these with the same `seed=42` reproduces the *exact*
Fold 4 validation set used during training, since `StratifiedKFold(shuffle=True, random_state=42)` is
deterministic. No separate CSV of fold membership is needed or was saved.


In [2]:
NOTEBOOK_DIR = Path.cwd()
CLASSIFICATION_DIR = NOTEBOOK_DIR.parent  # classification/datapreparepipeline/
sys.path.insert(0, str(CLASSIFICATION_DIR))
sys.path.insert(0, str(NOTEBOOK_DIR))

from trainer_engine import build_efficientnet_b0  # noqa: E402
from dataset import get_eval_transforms, IMAGE_SIZE  # noqa: E402
from cv_dataset import load_cv_pool, build_folds, PatientListDataset, IMAGES_DIR, SEED, N_FOLDS  # noqa: E402

print("IMAGES_DIR:", IMAGES_DIR)
print("IMAGE_SIZE:", IMAGE_SIZE, " SEED:", SEED, " N_FOLDS:", N_FOLDS)


IMAGES_DIR: D:\khaje\EYES-DEFY-ANEMIA\classification\data\processed\images\forniceal_palpebral
IMAGE_SIZE: 256  SEED: 42  N_FOLDS: 5


In [3]:
# --------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------
FOLD_NUM = 4  # 1-indexed, matches the checkpoint filename and history JSON naming
DROPOUT_RATE = 0.2  # matches training (irrelevant at inference: Dropout is inactive under model.eval())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = NOTEBOOK_DIR / "outputs" / "checkpoints" / f"efficientnet_b0_forniceal_palpebral_cv_fold{FOLD_NUM}_best.pth"
GRADCAM_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "gradcam"
GRADCAM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROB_THRESHOLD = 0.5  # matches compute_metrics()'s convention throughout this project

print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT_PATH, " exists:", CHECKPOINT_PATH.exists())
print("Grad-CAM figures will be saved to:", GRADCAM_OUTPUT_DIR)


Device: cuda
Checkpoint: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\checkpoints\efficientnet_b0_forniceal_palpebral_cv_fold4_best.pth  exists: True
Grad-CAM figures will be saved to: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam


## 2. Model & weights loading (Fold 4)


In [4]:
model = build_efficientnet_b0(DROPOUT_RATE)

state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)  # strict=True by default -- shapes must match exactly

model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Loaded fold {FOLD_NUM} checkpoint. Total params: {n_params:,}  Trainable (head only): {n_trainable:,}")


Loaded fold 4 checkpoint. Total params: 4,008,829  Trainable (head only): 1,281


## 3. Grad-CAM target layer

**Important gotcha specific to this model:** the backbone is entirely frozen (`requires_grad=False` on every
backbone parameter, via `trainer_engine._freeze_all`). By default, PyTorch's autograd only builds a
computation graph for tensors that trace back to something requiring gradients — with a frozen backbone
*and* a plain input tensor (which doesn't require grad by default), the intermediate feature maps we want to
hook into would **never actually get a gradient computed**, producing silently blank/`None` Grad-CAM heatmaps
with no error raised. The fix, applied in `GradCAM.generate()` below, is to explicitly set
`input_tensor.requires_grad_(True)` before the forward pass — this makes autograd track the full forward
path end-to-end regardless of whether the *parameters* along that path require grad, since graph-building is
driven by whether any input to an op requires grad, not by the op's own parameters.

**Target layer resolution:** for a `torchvision`-style model (this project's convention, confirmed in
`trainer_engine.py` — not a `timm` model), the final convolutional feature map is `model.features[-1]` — for
EfficientNet-B0 this is a `Conv2dNormActivation` block (`Conv2d -> BatchNorm2d -> SiLU`) expanding to 1280
channels at an 8x8 spatial resolution (256 / 32, since EfficientNet-B0 downsamples by 32x total). This is the
standard Grad-CAM target for EfficientNet: the last spatially-resolved, semantically-rich feature map before
global pooling. A generic fallback (last `nn.Conv2d` anywhere in the model) is included so this cell doesn't
silently break if pointed at a differently-structured (e.g. `timm`) checkpoint in the future.


In [5]:
def find_target_layer(model: nn.Module) -> nn.Module:
    """Automatically resolves the final convolutional feature layer for Grad-CAM."""
    if hasattr(model, "features"):
        # torchvision-style (efficientnet_b0, densenet121, convnext_tiny, ...)
        layer = model.features[-1]
        print(f"[find_target_layer] torchvision-style model -- using model.features[-1]: {layer.__class__.__name__}")
        return layer

    # Generic fallback: last nn.Conv2d anywhere in the model (covers non-torchvision / timm-style models).
    last_conv, last_name = None, None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_conv, last_name = module, name
    if last_conv is None:
        raise RuntimeError("No nn.Conv2d layer found anywhere in the model -- cannot attach Grad-CAM hooks.")
    print(f"[find_target_layer] Fallback -- using last nn.Conv2d layer found: '{last_name}'")
    return last_conv


target_layer = find_target_layer(model)
print("Resolved target layer:", target_layer)

# Verify with a dummy forward pass -- confirms the spatial resolution/channel count we'll be working with.
with torch.no_grad():
    _dummy_out = model.features(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE))
print("Target feature map shape [B, C, H, W]:", tuple(_dummy_out.shape))


[find_target_layer] torchvision-style model -- using model.features[-1]: Conv2dNormActivation
Resolved target layer: Conv2dNormActivation(
  (0): Conv2d(320, 1280, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (1): BatchNorm2d(1280, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): SiLU(inplace=True)
)


Target feature map shape [B, C, H, W]: (1, 1280, 8, 8)


In [6]:
class GradCAM:
    """Minimal, dependency-free Grad-CAM (Selvaraju et al., 2017) via forward/backward hooks.

    weight_k = global-average-pool(dScore/dA_k)   -- one scalar weight per activation channel
    CAM      = ReLU( sum_k weight_k * A_k )        -- weighted combination of activation maps
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor: torch.Tensor) -> tuple:
        """input_tensor: [1, 3, H, W], already normalized/on-device.
        Returns (cam [H_feat, W_feat] in [0, 1], logit: float, prob: float)."""
        self.model.zero_grad(set_to_none=True)

        # Required even though the backbone is frozen -- see the markdown cell above for why.
        input_tensor = input_tensor.clone().requires_grad_(True)

        logit = self.model(input_tensor)  # [1, 1], raw logit (no Sigmoid in the model)
        score = logit[0, 0]               # backprop from the raw logit, not the sigmoid probability
        score.backward()

        # weight_k = GAP over spatial dims of dScore/dA_k, for each of the C channels
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)      # [1, C, 1, 1]
        cam = (weights * self.activations).sum(dim=1, keepdim=False)  # [1, H_feat, W_feat]
        cam = torch.relu(cam)[0]                                      # [H_feat, W_feat]

        cam_min, cam_max = cam.min(), cam.max()
        cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)  # normalize to [0, 1]

        prob = torch.sigmoid(logit).item()
        return cam.cpu().numpy(), logit.item(), prob


gradcam = GradCAM(model, target_layer)
print("GradCAM initialized on:", target_layer.__class__.__name__)


GradCAM initialized on: Conv2dNormActivation


## 4. Targeted data extraction — reconstruct Fold 4's validation set and run inference

Rebuilds the *exact* Fold 4 train/val split used during training (same pool, same `StratifiedKFold(seed=42)`),
then runs every Fold-4 validation patient through the loaded checkpoint once (no augmentation — the same
`get_eval_transforms()` pipeline validation used during training) to get each patient's predicted probability
and label, so we can filter into the India-FP / India-TP / Italy groups.


In [7]:
pool = load_cv_pool()
folds = build_folds(pool, n_folds=N_FOLDS, seed=SEED)
train_df, val_df = folds[FOLD_NUM - 1]  # FOLD_NUM is 1-indexed

print(f"Fold {FOLD_NUM}: train={len(train_df)}  val={len(val_df)}")
print("Val composition (country x anemic_label):")
print(val_df.groupby(["country", "anemic_label"]).size())


Fold 4: train=143  val=35
Val composition (country x anemic_label):
country  anemic_label
India    0.0              5
         1.0             11
Italy    0.0             15
         1.0              4
dtype: int64


In [8]:
eval_transform = get_eval_transforms(IMAGE_SIZE)


def load_model_input(patient_id: str) -> torch.Tensor:
    """Loads and preprocesses one patient's crop exactly as PatientListDataset does -- [1, 3, H, W] on DEVICE."""
    image = np.array(Image.open(IMAGES_DIR / f"{patient_id}.jpg").convert("RGB"))
    tensor = eval_transform(image=image)["image"]  # [3, H, W], normalized
    return tensor.unsqueeze(0).to(DEVICE)


def load_display_image(patient_id: str) -> np.ndarray:
    """Loads the same crop for human viewing -- resized to match the model's spatial frame, NOT normalized."""
    image = Image.open(IMAGES_DIR / f"{patient_id}.jpg").convert("RGB")
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
    return np.array(image)  # uint8 [H, W, 3], RGB


records = []
with torch.no_grad():
    for _, row in val_df.iterrows():
        x = load_model_input(row["patient_id"])
        logit = model(x)
        prob = torch.sigmoid(logit).item()
        pred_label = int(prob > PROB_THRESHOLD)
        records.append({
            "patient_id": row["patient_id"],
            "country": row["country"],
            "true_label": int(row["anemic_label"]),
            "pred_prob": prob,
            "pred_label": pred_label,
        })

val_results = pd.DataFrame(records)
val_results.to_csv(GRADCAM_OUTPUT_DIR / f"fold{FOLD_NUM}_val_predictions.csv", index=False)
val_results


,patient_id,country,true_label,pred_prob,pred_label
0,India_013,India,1,0.565900,1
1,India_018,India,1,0.527759,1
2,India_027,India,0,0.696403,1
3,India_028,India,1,0.551884,1
4,India_030,India,1,0.512361,1
5,India_032,India,1,0.639807,1
6,India_056,India,1,0.629977,1
7,India_061,India,1,0.627643,1
8,India_063,India,1,0.632060,1
9,India_064,India,1,0.590734,1


In [9]:
# Sanity check against CV_5FOLD_ANALYSIS_REPORT.md's Fold 4 India confusion matrix [[TN=2,FP=3],[FN=0,TP=11]]
india = val_results[val_results["country"] == "India"]
italy = val_results[val_results["country"] == "Italy"]

india_fp = india[(india["true_label"] == 0) & (india["pred_label"] == 1)]
india_tp = india[(india["true_label"] == 1) & (india["pred_label"] == 1)]
india_tn = india[(india["true_label"] == 0) & (india["pred_label"] == 0)]
india_fn = india[(india["true_label"] == 1) & (india["pred_label"] == 0)]

italy_baseline = italy[italy["true_label"] == italy["pred_label"]]  # correctly classified -- "normal behavior"

print(f"India: TN={len(india_tn)} FP={len(india_fp)} FN={len(india_fn)} TP={len(india_tp)}"
      "  (expect TN=2 FP=3 FN=0 TP=11 from the CV report -- confirms this reconstruction matches training)")
print(f"Italy: n={len(italy)}, correctly classified (baseline group)={len(italy_baseline)}")

print("\nIndia False Positives (primary target):")
print(india_fp[["patient_id", "pred_prob"]].to_string(index=False))
print("\nIndia True Positives (control):")
print(india_tp[["patient_id", "pred_prob"]].to_string(index=False))
print("\nItaly baseline (correctly classified, sample):")
print(italy_baseline[["patient_id", "true_label", "pred_prob"]].head(4).to_string(index=False))


India: TN=2 FP=3 FN=0 TP=11  (expect TN=2 FP=3 FN=0 TP=11 from the CV report -- confirms this reconstruction matches training)
Italy: n=19, correctly classified (baseline group)=14

India False Positives (primary target):
patient_id  pred_prob
 India_027   0.696403
 India_083   0.696364
 India_088   0.532093

India True Positives (control):
patient_id  pred_prob
 India_013   0.565900
 India_018   0.527759
 India_028   0.551884
 India_030   0.512361
 India_032   0.639807
 India_056   0.629977
 India_061   0.627643
 India_063   0.632060
 India_064   0.590734
 India_084   0.647406
 India_094   0.674679

Italy baseline (correctly classified, sample):
patient_id  true_label  pred_prob
 Italy_012           1   0.652390
 Italy_027           1   0.642111
 Italy_029           1   0.581606
 Italy_043           0   0.314792


## 5. Visualization

For each selected patient: `[Original crop] | [Raw Grad-CAM heatmap] | [Superimposed overlay]`, titled with
cohort, true label, and predicted probability. Figures are displayed inline **and** saved to
`outputs/gradcam/` so they persist for the thesis writeup.


In [10]:
def visualize_gradcam(row: pd.Series, group_name: str, save: bool = True) -> None:
    patient_id = row["patient_id"]
    true_label = "Anemic" if row["true_label"] == 1 else "Non-anemic"
    pred_label = "Anemic" if row["pred_label"] == 1 else "Non-anemic"

    original = load_display_image(patient_id)  # uint8 [256, 256, 3] RGB

    input_tensor = load_model_input(patient_id)
    cam, logit, prob = gradcam.generate(input_tensor)  # cam: [H_feat, W_feat] in [0, 1]

    cam_resized = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
    heatmap_bgr = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(original, 0.55, heatmap_rgb, 0.45, 0)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    axes[0].imshow(original)
    axes[0].set_title("Original crop")
    axes[0].axis("off")

    axes[1].imshow(cam_resized, cmap="jet")
    axes[1].set_title("Raw Grad-CAM heatmap")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title("Superimposed")
    axes[2].axis("off")

    fig.suptitle(
        f"{group_name} -- patient {patient_id} -- Cohort: {row['country']} -- "
        f"True: {true_label} -- Predicted: {pred_label} (P(anemic)={prob:.3f})",
        fontsize=11,
    )
    fig.tight_layout()

    if save:
        safe_group = group_name.lower().replace(" ", "_")
        out_path = GRADCAM_OUTPUT_DIR / f"fold{FOLD_NUM}_{safe_group}_{patient_id}.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {out_path}")

    plt.show()
    plt.close(fig)


### 5.1 India False Positives (primary target)

The core diagnostic question: does attention sit on plausible conjunctival tissue (calibration hypothesis) or
on borders/corners/diffuse non-tissue regions (shortcut hypothesis)?


In [11]:
for _, row in india_fp.iterrows():
    visualize_gradcam(row, "India False Positive")


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_india_false_positive_India_027.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_30828\1120648555.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_india_false_positive_India_083.png


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_india_false_positive_India_088.png


### 5.2 India True Positives (control group)

Same cohort, correct prediction — compare attention patterns against the false positives above. If attention
looks similar regardless of correctness, that itself is evidence the model isn't discriminating much at all
within India (consistent with the near-constant "predict anemic" pattern already found in
`CV_5FOLD_ANALYSIS_REPORT.md` — recall was exactly 1.000 in all 5 folds).


In [12]:
# Capped to the same count as the FP group for a balanced side-by-side comparison.
for _, row in india_tp.head(len(india_fp)).iterrows():
    visualize_gradcam(row, "India True Positive")


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_india_true_positive_India_013.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_30828\1120648555.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_india_true_positive_India_018.png


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_india_true_positive_India_028.png


### 5.3 Italy baseline (correctly classified — normal model behavior)


In [13]:
for _, row in italy_baseline.head(4).iterrows():
    visualize_gradcam(row, "Italy Baseline")


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_italy_baseline_Italy_012.png


C:\Users\Asus\AppData\Local\Temp\ipykernel_30828\1120648555.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_italy_baseline_Italy_027.png


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_italy_baseline_Italy_029.png


Saved: D:\khaje\EYES-DEFY-ANEMIA\classification\datapreparepipeline\efficientnet_b0_forniceal_5fold_cv\outputs\gradcam\fold4_italy_baseline_Italy_043.png


## 6. How to read these results

- **Attention concentrated on central tissue, consistent shape/location across patients** → the model is at
  least *looking at* the right anatomical region; the low India AUC is more likely a calibration/threshold
  problem (`pos_weight` + tiny validation samples) than a learned visual shortcut. In that case, DANN is
  probably the wrong tool — a per-country threshold recalibration or `pos_weight` adjustment is cheaper and
  more directly targeted.
- **Attention on image borders, corners, uniform/non-tissue regions, or wildly inconsistent across patients**
  → direct visual evidence supporting the shortcut-learning hypothesis (camera/illumination/compression
  artifacts, since these `forniceal_palpebral` crops contain no skin or sclera for the model to misuse
  instead). This would support proceeding with the CLAHE-on-V + DANN mitigation plan discussed earlier —
  ideally CLAHE first, re-measuring the domain gap with a lightweight discriminator, before deciding whether
  DANN has meaningful residual work left to do (per the earlier theoretical consultation's staging
  recommendation).
- **India FP vs. India TP attention patterns look near-identical** → reinforces that the model isn't doing
  much genuine per-patient discrimination within India at all (consistent with the CV report's finding of
  exactly 1.000 recall replicated across all 5 independent folds).

All figures are saved under `outputs/gradcam/` alongside `fold4_val_predictions.csv` (every Fold-4 validation
patient's true label, predicted probability, and predicted label) for reference in the thesis writeup.


---
# 7. Quantifying the Grad-CAM evidence: tissue-attention ratio

Turns the visual "does attention sit on tissue or background" impression from Section 5 into a number: for
each patient, what fraction of the Grad-CAM heatmap's total activation energy falls inside the segmented
tissue region vs. the padding background. Aggregated (mean/std) across India-FP / India-TP / Italy-baseline,
this either confirms or refutes the shortcut-learning hypothesis with hard numbers, per the theoretical
consultation's Step 1 recommendation.

**Important correction found while building this section (2026-08-01):** a naive "background = near-black"
threshold is wrong for a real subset of this dataset. See 7.1.


## 7.1 Data-quality finding: two padding conventions, and they correlate with country

`prepare_dataset.py`'s `flatten_to_black()` alpha-composites each source crop onto a black canvas using the
crop's own alpha channel — this only produces a clean black background if alpha correctly encodes
transparency. The segmentation phase's *independent* pipeline already discovered and fixed a real source-data
issue where a subset of crops instead use an **opaque white background** (root `CLAUDE.md` Sec 1.4.4) —
`alpha_composite` doesn't fix that case, since the "background" pixels are fully opaque and get composited
as-is. `classification/datapreparepipeline/prepare_dataset.py` is a from-scratch reimplementation (isolation
rule, `03_tech_stack_and_rules.md`) and evidently never received an equivalent fix.

The cell below scans every processed `forniceal_palpebral` image directly (>30% of the 256x256 canvas
near-white is used as the flag, well above what a real tissue crop's own specular highlights would produce)
to confirm this is real and check whether it happens to correlate with country — which matters a great deal
for interpreting *any* India-vs-Italy comparison, Grad-CAM included.


In [14]:
import pandas as pd

FORNICEAL_DIR = CLASSIFICATION_DIR.parent / "data" / "processed" / "images" / "forniceal_palpebral"

bg_scan = []
for img_path in sorted(FORNICEAL_DIR.glob("*.jpg")):
    pid = img_path.stem
    arr = np.array(Image.open(img_path).convert("RGB"))
    gray_frac_white = (arr.mean(axis=2) > 240).mean()
    bg_scan.append({"patient_id": pid, "country": pid.split("_")[0], "near_white_frac": gray_frac_white})

bg_df = pd.DataFrame(bg_scan)
bg_df["white_bg_flag"] = bg_df["near_white_frac"] > 0.30

print(f"Total patients scanned: {len(bg_df)}")
print(f"White-background flagged: {bg_df['white_bg_flag'].sum()} / {len(bg_df)}\n")
print("By country:")
print(bg_df.groupby("country")["white_bg_flag"].agg(["sum", "count", "mean"]))

WHITE_BG_PATIENT_IDS = set(bg_df.loc[bg_df["white_bg_flag"], "patient_id"])
print(f"\n{len(WHITE_BG_PATIENT_IDS)} flagged patients (all should be Italy_*):")
print(sorted(WHITE_BG_PATIENT_IDS))


Total patients scanned: 211
White-background flagged: 30 / 211

By country:
         sum  count      mean
country                      
India      0     95  0.000000
Italy     30    116  0.258621

30 flagged patients (all should be Italy_*):
['Italy_002', 'Italy_003', 'Italy_004', 'Italy_005', 'Italy_006', 'Italy_007', 'Italy_008', 'Italy_009', 'Italy_010', 'Italy_011', 'Italy_012', 'Italy_013', 'Italy_014', 'Italy_015', 'Italy_016', 'Italy_017', 'Italy_018', 'Italy_019', 'Italy_020', 'Italy_021', 'Italy_022', 'Italy_023', 'Italy_024', 'Italy_025', 'Italy_026', 'Italy_027', 'Italy_028', 'Italy_029', 'Italy_030', 'Italy_031']


## 7.2 Robust tissue mask (handles both padding conventions)

Detects which convention a given image uses (via the same near-white-fraction trigger as the scan above),
thresholds accordingly, then cleans the raw threshold up with morphological open/close and keeps only the
single largest connected component — so small compression-artifact specks or anti-aliased boundary noise
can't fragment the mask or get mistaken for a second tissue region.


In [15]:
def get_tissue_mask(
    image_rgb: np.ndarray,
    black_threshold: int = 15,
    white_threshold: int = 240,
    white_frac_trigger: float = 0.30,
) -> np.ndarray:
    """Boolean mask, True = tissue, False = padding background. Handles both the standard
    black-padding convention and the white-padding convention found in Sec. 7.1."""
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    near_white_frac = (gray > white_threshold).mean()

    if near_white_frac > white_frac_trigger:
        # White-background convention -- tissue is the NON-white region.
        _, raw_mask = cv2.threshold(gray, white_threshold, 255, cv2.THRESH_BINARY_INV)
    else:
        # Standard black-background convention -- tissue is the NON-black region.
        _, raw_mask = cv2.threshold(gray, black_threshold, 255, cv2.THRESH_BINARY)

    raw_mask = raw_mask.astype(np.uint8)

    # Morphological cleanup: close small internal holes (e.g. specular highlights on wet
    # tissue), remove small stray specks (e.g. compression artifacts in the background).
    kernel = np.ones((5, 5), np.uint8)
    cleaned = cv2.morphologyEx(raw_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel, iterations=1)

    # Keep only the largest connected component -- the tissue blob itself.
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(cleaned, connectivity=8)
    if n_labels <= 1:
        return np.zeros_like(gray, dtype=bool)
    largest_label = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return labels == largest_label


In [16]:
# Visual sanity check before trusting this for the metric: one black-bg patient, one white-bg patient.
_check_ids = ["India_027", "Italy_012"]  # Italy_012 is white-bg-flagged (Sec 7.1)
fig, axes = plt.subplots(len(_check_ids), 2, figsize=(8, 4 * len(_check_ids)))
for i, pid in enumerate(_check_ids):
    img = load_display_image(pid)
    mask = get_tissue_mask(img)
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"{pid} -- original")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title(f"{pid} -- tissue mask ({mask.mean()*100:.1f}% of canvas)")
    axes[i, 1].axis("off")
fig.tight_layout()
plt.show()
plt.close(fig)


C:\Users\Asus\AppData\Local\Temp\ipykernel_30828\3895873354.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7.3 Tissue-attention ratio metric

`ratio = sum(cam * tissue_mask) / sum(cam)` — the fraction of the heatmap's total activation energy that
falls inside the tissue region. Scale-invariant to the per-image min-max normalization already applied in
`GradCAM.generate()` (both numerator and denominator scale by the same constant), so ratios are directly
comparable across patients.

**Sample size note:** this reuses the *full* `india_fp`, `india_tp`, and `italy_baseline` groups already
computed in Section 4 — not just the capped subsets that were visualized in Section 5 (that cap was for
display brevity only). India FP is fixed at n=3 (that's every false positive that exists in Fold 4's
validation set); India TP uses all 11; Italy baseline uses all 14 correctly-classified Italy patients. Given
n=3 for the primary group, treat this as a first, suggestive read — not a statistically powered test.


In [17]:
def tissue_attention_ratio(patient_id: str) -> float:
    original = load_display_image(patient_id)
    mask = get_tissue_mask(original)

    input_tensor = load_model_input(patient_id)
    cam, _logit, _prob = gradcam.generate(input_tensor)
    cam_resized = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)

    total_energy = cam_resized.sum()
    if total_energy <= 1e-8:
        return float("nan")
    tissue_energy = cam_resized[mask].sum()
    return float(tissue_energy / total_energy)


groups = {
    "India False Positive": india_fp,
    "India True Positive": india_tp,
    "Italy Baseline": italy_baseline,
}

ratio_records = []
for group_name, group_df in groups.items():
    for _, row in group_df.iterrows():
        ratio = tissue_attention_ratio(row["patient_id"])
        ratio_records.append({
            "group": group_name,
            "patient_id": row["patient_id"],
            "white_bg": row["patient_id"] in WHITE_BG_PATIENT_IDS,
            "tissue_attention_ratio": ratio,
        })

ratio_df = pd.DataFrame(ratio_records)
ratio_df.to_csv(GRADCAM_OUTPUT_DIR / f"fold{FOLD_NUM}_tissue_attention_ratios.csv", index=False)
ratio_df


,group,patient_id,white_bg,tissue_attention_ratio
0,India False Positive,India_027,False,0.209318
1,India False Positive,India_083,False,0.220846
2,India False Positive,India_088,False,0.032037
3,India True Positive,India_013,False,0.235454
4,India True Positive,India_018,False,0.051920
5,India True Positive,India_028,False,0.227473
6,India True Positive,India_030,False,0.323268
7,India True Positive,India_032,False,0.239787
8,India True Positive,India_056,False,0.255363
9,India True Positive,India_061,False,0.270529


In [18]:
summary = ratio_df.groupby("group")["tissue_attention_ratio"].agg(
    n="count", mean="mean", std="std", var="var", min="min", max="max"
)
# Preserve a fixed, meaningful group order rather than alphabetical.
summary = summary.reindex(["India False Positive", "India True Positive", "Italy Baseline"])

print("Tissue-attention ratio -- mean / std / variance by group:\n")
print(summary.to_string(float_format=lambda x: f"{x:.4f}"))

n_white_bg_in_sample = int(ratio_df["white_bg"].sum())
print(f"\n({n_white_bg_in_sample} of {len(ratio_df)} sampled patients use the white-background "
      f"convention -- correctly handled by get_tissue_mask(), flagged per-patient in the CSV/table above "
      f"for transparency.)")


Tissue-attention ratio -- mean / std / variance by group:

                       n   mean    std    var    min    max
group                                                      
India False Positive   3 0.1541 0.1058 0.0112 0.0320 0.2208
India True Positive   11 0.2579 0.1044 0.0109 0.0519 0.4756
Italy Baseline        14 0.1711 0.2078 0.0432 0.0000 0.7039

(3 of 28 sampled patients use the white-background convention -- correctly handled by get_tissue_mask(), flagged per-patient in the CSV/table above for transparency.)


In [19]:
fig, ax = plt.subplots(figsize=(7, 5))
order = ["India False Positive", "India True Positive", "Italy Baseline"]
data_by_group = [ratio_df[ratio_df["group"] == g]["tissue_attention_ratio"].values for g in order]

bp = ax.boxplot(data_by_group, tick_labels=order, showmeans=True)
for i, vals in enumerate(data_by_group, start=1):
    jitter = np.random.default_rng(0).normal(0, 0.03, size=len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals, alpha=0.7, color="tab:blue", zorder=3)

ax.set_ylabel("Tissue-attention ratio")
ax.set_title(f"Fold {FOLD_NUM} -- Grad-CAM tissue-attention ratio by group")
ax.set_ylim(-0.02, 1.02)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(GRADCAM_OUTPUT_DIR / f"fold{FOLD_NUM}_tissue_attention_ratio_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)


C:\Users\Asus\AppData\Local\Temp\ipykernel_30828\1132370619.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7.4 Reading these numbers

- **Lower tissue-attention ratio for India-FP than for India-TP and/or Italy-Baseline** → supports the
  shortcut hypothesis: the model's attention systematically leaks outside the tissue region specifically when
  it gets India patients wrong, not just in general.
- **All three groups similar** → the shortcut hypothesis is not well-supported by attention *location* alone;
  the calibration/threshold explanation (small samples + `pos_weight`) becomes relatively more credible, and
  cheaper fixes (per-country threshold recalibration, `pos_weight` adjustment) should be tried before CLAHE/DANN.
- **Whatever the result, it should be re-read in light of Sec. 7.1**: Italy-Baseline is the only group with
  any white-background patients in it (India has none, by construction — 0/95 in the full dataset). If
  Italy's tissue-attention ratio comes out high partly *because* white-background images make the tissue
  region trivially easy to attend to precisely (small, high-contrast blob against a flat field), that's a
  confound in this diagnostic itself, not evidence the model has learned something meaningfully different
  between countries. The `white_bg` column in `ratio_df` is there to let this be checked directly (e.g.
  compare Italy-Baseline's white-bg vs. black-bg subsets against each other before comparing Italy to India
  at all).
- **Separately from the DANN/CLAHE question**: Sec. 7.1's finding is arguably the more urgent one on its own
  merits. A 26%-of-Italy, 0%-of-India, image-background-color confound baked directly into the training data
  is a data bug with a direct fix (port the segmentation phase's white-background detection/handling into
  `prepare_dataset.py`, reprocess the affected 30 patients, retrain) — worth fixing and re-measuring before
  investing in any training-time mitigation for a confound that may partly be an artifact of this bug rather
  than a property of the real camera/hospital differences CLAHE and DANN are meant to address.
